# Experiment F: Scoped Automatic Feature Selection

Experiments A and B both reach a ROC-AUC around 0.78-0.79, but precision and F1 stay low. This experiment checks whether the feature representation itself is the limiting factor, by letting an L1-regularized model choose which variables matter instead of hand-picking them like A and B did.

This is a **scoped** automatic-selection experiment. It does not search over all 216 candidate variables identified during the dataset audit. Most of those variables do not yet have documented special-code handling, so including them would mean guessing at CCHS codes instead of using verified ones.

Instead, this experiment uses the 16 variables that already have documented special-code handling from Experiments A, B, C, and D1/D2 (general-population compatible only). `HWTDGISW` and `HWTDGWHO` are left out as separate variables since they are the age-specific inputs already combined into `BMI_CLASS`.

In [1]:
import pandas as pd
import numpy as np

pumf = pd.read_csv("../Data_Données/pumf_cchs.csv")

print("PUMF shape:", pumf.shape)

PUMF shape: (67079, 255)


In [2]:
model_data = pumf[pumf["CCC_05"].isin([1, 2])].copy()

model_data["target"] = (model_data["CCC_05"] == 1).astype(int)

print("Modelling population:", model_data.shape[0])
print(model_data["target"].value_counts().sort_index())

Modelling population: 66242
target
0    60248
1     5994
Name: count, dtype: int64


## Candidate feature pool

16 variables, all with special-code handling already established in earlier experiments.

In [3]:
CANDIDATE_FEATURES = [
    "DHHGAGE", "DHH_SEX", "DHHDGHSZ", "EDDVH3", "GEOGPRV", "INCDGHH",
    "FSCDVHF2", "ALCDVTTM", "ECV_05", "SDCDGIMM", "LSM_01", "WTP_50",
    "SMKDVSTY", "CCC_80", "CCC_90", "BMI_CLASS"
]

print("Number of candidate variables:", len(CANDIDATE_FEATURES))
print(CANDIDATE_FEATURES)

Number of candidate variables: 16
['DHHGAGE', 'DHH_SEX', 'DHHDGHSZ', 'EDDVH3', 'GEOGPRV', 'INCDGHH', 'FSCDVHF2', 'ALCDVTTM', 'ECV_05', 'SDCDGIMM', 'LSM_01', 'WTP_50', 'SMKDVSTY', 'CCC_80', 'CCC_90', 'BMI_CLASS']


In [4]:
# Same BMI harmonization as Experiment B: youth and adult BMI codes come
# from different source variables, so they are combined into one column.
model_data["BMI_CLASS"] = np.nan

youth_mask = model_data["DHHGAGE"] == 1
adult_mask = model_data["DHHGAGE"].isin([2, 3, 4, 5])

model_data.loc[youth_mask, "BMI_CLASS"] = model_data.loc[youth_mask, "HWTDGWHO"]
model_data.loc[adult_mask, "BMI_CLASS"] = model_data.loc[adult_mask, "HWTDGISW"]

print(model_data["BMI_CLASS"].value_counts(dropna=False).sort_index())

BMI_CLASS
1.0    26053
2.0    37045
6.0       32
9.0     3112
Name: count, dtype: int64


In [5]:
SPECIAL_CODES = {
    "DHHDGHSZ": [9],
    "DHHGAGE": [],
    "DHH_SEX": [],
    "EDDVH3": [9],
    "GEOGPRV": [],
    "INCDGHH": [9],
    "FSCDVHF2": [9],
    "ALCDVTTM": [9],
    "ECV_05": [9],
    "SDCDGIMM": [9],
    "LSM_01": [99],
    "WTP_50": [9],
    "SMKDVSTY": [96, 99],
    "CCC_80": [9],
    "CCC_90": [9],
    "BMI_CLASS": [6, 9]
}

def apply_special_codes(df, special_codes):
    result = df.copy()

    for column, codes in special_codes.items():
        if column in result.columns:
            result[column] = result[column].replace(codes, np.nan)

    return result

clean_model_data = apply_special_codes(model_data, SPECIAL_CODES)

print("Missing values after special-code handling:")
print(clean_model_data[CANDIDATE_FEATURES].isna().sum())

Missing values after special-code handling:
DHHGAGE         0
DHH_SEX         0
DHHDGHSZ      486
EDDVH3       2276
GEOGPRV         0
INCDGHH       947
FSCDVHF2     1513
ALCDVTTM      303
ECV_05        141
SDCDGIMM      835
LSM_01        702
WTP_50       1258
SMKDVSTY     5771
CCC_80        494
CCC_90        155
BMI_CLASS    3144
dtype: int64


## Train / validation / test split

Same population, same seed, same stratified two-step split used in A and B.

In [6]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_val_idx, test_idx = train_test_split(
    model_data.index,
    test_size=0.20,
    stratify=model_data["target"],
    random_state=RANDOM_STATE
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.20,
    stratify=model_data.loc[train_val_idx, "target"],
    random_state=RANDOM_STATE
)

print("Training:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Training: 42394
Validation: 10599
Test: 13249


In [7]:
X_train = clean_model_data.loc[train_idx, CANDIDATE_FEATURES]
X_val = clean_model_data.loc[val_idx, CANDIDATE_FEATURES]
X_test = clean_model_data.loc[test_idx, CANDIDATE_FEATURES]

y_train = model_data.loc[train_idx, "target"]
y_val = model_data.loc[val_idx, "target"]
y_test = model_data.loc[test_idx, "target"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (42394, 16) (42394,)
Validation: (10599, 16) (10599,)
Test: (13249, 16) (13249,)


## Preprocessing

Same approach as A and B: `LSM_01` is numeric (imputed and scaled), everything else is categorical (imputed and one-hot encoded). Fitted on the training data only.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

NUMERIC_FEATURES = ["LSM_01"]
CATEGORICAL_FEATURES = [f for f in CANDIDATE_FEATURES if f not in NUMERIC_FEATURES]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (42394, 55)
Processed validation shape: (10599, 55)
Processed test shape: (13249, 55)


## L1 feature selection

An L1-regularized logistic regression shrinks weak coefficients to exactly zero, so it can be used to trim variables in a data-driven way. It is fit on the training data only, so validation and test data never influence which variables get selected.

In [9]:
import warnings
from sklearn.linear_model import LogisticRegression

# liblinear still triggers a scikit-learn deprecation notice for penalty="l1",
# the fit itself is unaffected, so the warning is just suppressed here
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

l1_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)

l1_model.fit(X_train_processed, y_train)

print("L1 model trained.")

L1 model trained.


Each categorical variable becomes several one-hot columns. To map coefficients back to the original variables, we use the encoder's `categories_` (the number of columns it produced for each variable) instead of parsing column-name strings, since several variable names contain underscores.

In [10]:
categories = preprocessor.named_transformers_["categorical"].named_steps["encoder"].categories_
coefs = l1_model.coef_[0]

var_has_nonzero = {}
position = 0

# LSM_01 is the one numeric column, placed first by the ColumnTransformer
var_has_nonzero["LSM_01"] = bool(np.any(coefs[position:position + 1] != 0))
position += 1

for feature, feature_categories in zip(CATEGORICAL_FEATURES, categories):
    width = len(feature_categories)
    var_has_nonzero[feature] = bool(np.any(coefs[position:position + width] != 0))
    position += width

print("Coefficients accounted for:", position, "/", len(coefs))

Coefficients accounted for: 55 / 55


In [11]:
selected_features = [f for f in CANDIDATE_FEATURES if var_has_nonzero[f]]
dropped_features = [f for f in CANDIDATE_FEATURES if not var_has_nonzero[f]]

print("Original candidate count:", len(CANDIDATE_FEATURES))
print("Selected variable count:", len(selected_features))

print("\nSelected variables:")
print(selected_features)

print("\nDropped variables:")
print(dropped_features)

Original candidate count: 16
Selected variable count: 16

Selected variables:
['DHHGAGE', 'DHH_SEX', 'DHHDGHSZ', 'EDDVH3', 'GEOGPRV', 'INCDGHH', 'FSCDVHF2', 'ALCDVTTM', 'ECV_05', 'SDCDGIMM', 'LSM_01', 'WTP_50', 'SMKDVSTY', 'CCC_80', 'CCC_90', 'BMI_CLASS']

Dropped variables:
[]


## Final model on selected variables

A plain (non-penalized) logistic regression is refit using only the selected variables, following the same setup as A and B. This gives a clean, interpretable model rather than relying on the L1 model's regularized coefficients directly.

In [12]:
SELECTED_NUMERIC = [f for f in NUMERIC_FEATURES if f in selected_features]
SELECTED_CATEGORICAL = [f for f in CATEGORICAL_FEATURES if f in selected_features]

final_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, SELECTED_NUMERIC),
    ("categorical", categorical_pipeline, SELECTED_CATEGORICAL)
])

X_train_final = final_preprocessor.fit_transform(clean_model_data.loc[train_idx, selected_features])
X_val_final = final_preprocessor.transform(clean_model_data.loc[val_idx, selected_features])
X_test_final = final_preprocessor.transform(clean_model_data.loc[test_idx, selected_features])

print("Processed training shape:", X_train_final.shape)
print("Processed validation shape:", X_val_final.shape)
print("Processed test shape:", X_test_final.shape)

Processed training shape: (42394, 55)
Processed validation shape: (10599, 55)
Processed test shape: (13249, 55)


In [13]:
final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

final_model.fit(X_train_final, y_train)

print("Final model trained.")

Final model trained.


In [14]:
train_prob = final_model.predict_proba(X_train_final)[:, 1]
val_prob = final_model.predict_proba(X_val_final)[:, 1]
test_prob = final_model.predict_proba(X_test_final)[:, 1]

print("Predicted probabilities generated.")

Predicted probabilities generated.


## Threshold selection on validation

Same approach as Experiment A: sweep thresholds on the validation set and lock the one with the highest F1, before touching the test set.

In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

thresholds = np.arange(0.10, 0.91, 0.01)

threshold_results = []

for threshold in thresholds:
    val_pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0),
        "accuracy": accuracy_score(y_val, val_pred)
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[threshold_results["f1"].idxmax()]

FINAL_THRESHOLD = float(best_row["threshold"])
VAL_F1 = float(best_row["f1"])

print("Selected validation threshold:", round(FINAL_THRESHOLD, 3))
print(best_row.round(4))

Selected validation threshold: 0.71
threshold    0.7100
precision    0.3103
recall       0.5297
f1           0.3914
accuracy     0.8509
Name: 61, dtype: float64


## Test evaluation

Threshold is locked. The test set is evaluated once.

In [16]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, log_loss

final_test_pred = (test_prob >= FINAL_THRESHOLD).astype(int)

test_accuracy = accuracy_score(y_test, final_test_pred)
test_precision = precision_score(y_test, final_test_pred, zero_division=0)
test_recall = recall_score(y_test, final_test_pred, zero_division=0)
test_f1 = f1_score(y_test, final_test_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, test_prob)
test_pr_auc = average_precision_score(y_test, test_prob)

print("Final Test Results")
print(f"Threshold: {FINAL_THRESHOLD:.3f}")
print(f"Accuracy:  {test_accuracy:.3f}")
print(f"Precision: {test_precision:.3f}")
print(f"Recall:    {test_recall:.3f}")
print(f"F1-score:  {test_f1:.3f}")
print(f"ROC-AUC:   {test_roc_auc:.3f}")
print(f"PR-AUC:    {test_pr_auc:.3f}")

cm = confusion_matrix(y_test, final_test_pred)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion matrix:")
print(cm)

Final Test Results
Threshold: 0.710
Accuracy:  0.846
Precision: 0.298
Recall:    0.515
F1-score:  0.378
ROC-AUC:   0.827
PR-AUC:    0.317

Confusion matrix:
[[10593  1457]
 [  581   618]]


In [17]:
train_log_loss = log_loss(y_train, train_prob)
val_log_loss = log_loss(y_val, val_prob)
test_log_loss = log_loss(y_test, test_prob)

print("Log Loss")
print(f"Training:   {train_log_loss:.4f}")
print(f"Validation: {val_log_loss:.4f}")
print(f"Test:       {test_log_loss:.4f}")

Log Loss
Training:   0.5148
Validation: 0.5163
Test:       0.5134


## Save results

In [18]:
experiment_f_results = pd.DataFrame([{
    "experiment": "F_Scoped_Automatic_Selection",
    "model": "Logistic Regression",
    "original_candidate_count": len(CANDIDATE_FEATURES),
    "selected_variable_count": len(selected_features),
    "selected_variables": ", ".join(selected_features),
    "dropped_variables": ", ".join(dropped_features),
    "threshold": FINAL_THRESHOLD,
    "validation_f1": VAL_F1,
    "accuracy": test_accuracy,
    "precision": test_precision,
    "recall": test_recall,
    "f1": test_f1,
    "roc_auc": test_roc_auc,
    "pr_auc": test_pr_auc,
    "train_log_loss": train_log_loss,
    "validation_log_loss": val_log_loss,
    "test_log_loss": test_log_loss,
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
    "train_shape": str(X_train_final.shape),
    "validation_shape": str(X_val_final.shape),
    "test_shape": str(X_test_final.shape)
}])

experiment_f_results.to_csv("experiment_F_results.csv", index=False)

print("Saved: experiment_F_results.csv")
experiment_f_results

Saved: experiment_F_results.csv


,experiment,model,original_candidate_count,selected_variable_count,selected_variables,dropped_variables,threshold,validation_f1,accuracy,precision,...,train_log_loss,validation_log_loss,test_log_loss,true_negatives,false_positives,false_negatives,true_positives,train_shape,validation_shape,test_shape
0,F_Scoped_Automatic_Selection,Logistic Regression,16,16,"DHHGAGE, DHH_SEX, DHHDGHSZ, EDDVH3, GEOGPRV, I...",,0.71,0.391371,0.846177,0.297831,...,0.514829,0.516255,0.513376,10593,1457,581,618,"(42394, 55)","(10599, 55)","(13249, 55)"


## Results

At C=1.0, all 16 candidate variables kept a non-zero coefficient, so none were dropped by L1.

Final model (16 variables, threshold 0.710) on the untouched test set:

- Accuracy: 0.846, Precision: 0.298, Recall: 0.515, F1: 0.378
- ROC-AUC: 0.827, PR-AUC: 0.317
- Confusion matrix: TN 10593, FP 1457, FN 581, TP 618

This beats A (F1 0.309, ROC-AUC 0.782) and B (F1 0.336, ROC-AUC 0.792) on every metric above. Train/val/test log loss are close (0.515 / 0.516 / 0.513), so no overfitting.

Note: F also includes CCC_80 and CCC_90 (not in A/B), and since L1 dropped nothing, part of this gain may come from those two variables rather than the selection step itself. D1/D2 would isolate that.